# mit-incidentalita-mensile-2001-2018 - notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- fonte: MIT — incidentalità stradale mensile, 2001-2018 (216 mesi)
- 19 righe mensili con anomalie documentate in notes.md
- **non** sostituisce l'analisi pubblica


In [ ]:
import re
import yaml
import pandas as pd
from pathlib import Path

# --- Unici parametri da impostare manualmente ---
METRICA = "incidenti"  # colonna numerica principale nel mart
METRICA_CLEAN = "incidenti"  # colonna corrispondente nel clean

# --- Anchor: usa il path del notebook se disponibile (VSCode), altrimenti cwd ---
try:
    _start = Path(__vsc_ipynb_file__).resolve().parent  # VSCode Jupyter
except NameError:
    _start = Path.cwd().resolve()

# Walk up finché non troviamo dataset.yml
candidate_dir = None
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        candidate_dir = probe
        break

if candidate_dir is None:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())

SLUG = cfg["dataset"]["name"]
ANNO_RUN = cfg["dataset"]["years"][-1]  # placeholder: [2001] → copre 2001-2018
MART_TABLE = cfg["mart"]["tables"][0]["name"]
ENCODING = cfg.get("clean", {}).get("read", {}).get("encoding", "utf-8")
DELIM = cfg.get("clean", {}).get("read", {}).get("delim", ",")
HEADER = cfg.get("clean", {}).get("read", {}).get("header", True)
SKIP = cfg.get("clean", {}).get("read", {}).get("skip", 0)

DI_ROOT = (candidate_dir / cfg["root"]).resolve()
RAW_DIR = DI_ROOT / "data" / "raw" / SLUG / str(ANNO_RUN)
CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR = DI_ROOT / "data" / "mart" / SLUG / str(ANNO_RUN)
SQL_DIR = candidate_dir / "sql"

print(f"slug      : {SLUG}")
print(f"anno_run  : {ANNO_RUN}  (placeholder: file copre 2001-2018)")
print(f"mart table: {MART_TABLE}")
print(f"encoding  : {ENCODING}  delim: {repr(DELIM)}")
print(f"header    : {HEADER}  skip: {SKIP}")

slug      : mit_incidentalita_mensile
anno_run  : 2001  (placeholder: file copre 2001-2018)
mart table: mart_mensile
encoding  : utf-8  delim: ','
header    : False  skip: 1



## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.


In [ ]:
for sql_file in sorted(SQL_DIR.glob("**/*.sql")):
    print(f"{'=' * 60}")
    print(f"  {sql_file.relative_to(candidate_dir)}")
    print(f"{'=' * 60}")
    print(sql_file.read_text())
    print()

  sql/clean.sql
-- clean.sql — mit-incidentalita-mensile-2001-2018
--
-- Perimetro: solo righe mensili (216 righe su 288 totali).
-- Le 72 righe trimestrali ("1° Trimestre", ecc.) sono escluse perché
-- la fonte MIT presenta anomalie non recuperabili su ~13 trimestri:
-- trailing zero mancante e valori irregolari.
--
-- Note tecniche:
-- - il file raw è utf-8 con BOM: si legge con header:false + skip:1
--   e colonne esplicite (evita problemi di nome colonna con BOM)
-- - i campi percentuale/indice usano la virgola come decimale nel raw
--   (valori quoted): REPLACE(',', '.') + CAST AS DOUBLE
-- - il campo mese_raw ha leading spaces nel raw: TRIM
-- - alcune righe trimestrali tardive hanno colonne in eccesso (null_padding)

with base as (
    select
        trim(mese_raw)                                          as mese,
        cast(trim(anno_raw) as integer)                        as anno,
        cast(nullif(trim(incidenti_raw), '') as integer)       as incidenti,
        cast(nulli

## 1. Raw

Verifica che il file raw esista e sia leggibile.


In [ ]:
import duckdb

raw_files = (
    sorted(RAW_DIR.glob("*.csv"))
    + sorted(RAW_DIR.glob("*.xlsx"))
    + sorted(RAW_DIR.glob("*.parquet"))
)
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

# Usa duckdb per leggere il raw — gestisce correttamente i CSV con delimiter quoted
con = duckdb.connect()
raw_full = con.query(f"SELECT * FROM '{raw_path}'").fetchdf()

N_RAW = len(raw_full)
print(f"Righe raw   : {N_RAW}")
print(f"Colonne raw : {len(raw_full.columns)} -> {list(raw_full.columns)[:5]}...")
raw_full.head(3)

File: mit_incidentalita_mensile_2001_2018.csv  (30 KB)
Righe raw   : 288
Colonne raw : 1 -> ['Mese,Anno ,Incidenti,Morti,Feriti,Incidenti mortali,Percentuale incidenti,Percentuale incidenti mortali,Percentuale morti,Percentuale feriti,Indice di mortalità generico,Indice di gravità,Indice di lesività,Indice specifico di mortalità ,Indice specifico di incidentalità']...



## 2. Clean

Verifica schema e null. Il file raw contiene 288 righe (216 mensili + 72 trimestrali); la clean trattiene solo le 216 mensili.


In [ ]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)

Righe clean : 216
Colonne     : ['mese', 'anno', 'incidenti', 'morti', 'feriti', 'incidenti_mortali', 'perc_incidenti', 'perc_incidenti_mortali', 'perc_morti', 'perc_feriti', 'indice_mortalita', 'indice_gravita', 'indice_lesivita', 'indice_mortalita_spec', 'indice_incidentalita_spec']



In [ ]:
# N_RAW potrebbe non essere definito se il raw non è stato letto (errore tokenizing pandas)
# Usa duckdb per contare righe raw in quel caso
if "N_RAW" not in dir():
    con2 = duckdb.connect()
    N_RAW = con2.query(f"SELECT COUNT(*) FROM '{raw_path}'").fetchone()[0]

dropped = N_RAW - N_CLEAN
dropped_pct = dropped / N_RAW * 100 if N_RAW > 0 else 0

print(f"Righe raw         : {N_RAW}")
print(f"Righe clean       : {N_CLEAN}")
print(f"Righe filtrate    : {dropped} ({dropped_pct:.1f}%)")
print()
if dropped > 0:
    print("-> Le righe filtrate sono le 72 trimestrali (1-4 Trimestre) — attese.")
else:
    print("-> Nessuna riga filtrata.")

print("\nNull per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")

Righe raw         : 288
Righe clean       : 216
Righe filtrate    : 72 (25.0%)

-> Le righe filtrate sono le 72 trimestrali (1-4 Trimestre) — attese.

Null per colonna clean:
  nessuno



## 3. Mart

Verifica unicità sulle chiavi del GROUP BY, anni presenti, null e consistenza dei totali con il clean.


In [ ]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)

Righe mart  : 216
Colonne     : ['anno', 'mese', 'mese_num', 'incidenti', 'morti', 'feriti', 'incidenti_mortali', 'indice_mortalita', 'indice_gravita', 'indice_lesivita']



In [ ]:
# Estrai chiavi GROUP BY dalla mart.sql per il check di unicita.
# Il mart_mensile ha GROUP BY su (anno, mese_num) — ogni combinazione e' unica.
mart_sql_path = SQL_DIR / "mart" / f"{MART_TABLE}.sql"
groupby_keys = []
if mart_sql_path.exists():
    sql_text = mart_sql_path.read_text()
    # Cerca GROUP BY anche su piu righe
    match = re.search(
        r"group\s+by\s+([\w\s,]+?)(?:\s+order\s|\s+limit\s|;|$)",
        sql_text,
        re.IGNORECASE | re.DOTALL,
    )
    if match:
        keys_str = match.group(1).replace("\n", " ")
        groupby_keys = [k.strip().split(".")[-1] for k in keys_str.split(",")]
        groupby_keys = [k for k in groupby_keys if k in mart.columns]

if groupby_keys:
    dups = mart.duplicated(subset=groupby_keys).sum()
    print(f"Chiavi GROUP BY: {groupby_keys}")
    print(f"Duplicati      : {dups}")
    assert dups == 0, f"FAIL: {dups} righe duplicate sulle chiavi del mart"
    print("OK: nessun duplicato.")
else:
    print("GROUP BY non trovato — verificare manualmente.")
    # Fallback: verifica manuale che (anno, mese_num) sia unico
    dups = mart.duplicated(subset=["anno", "mese_num"]).sum()
    print(f"Duplicati su (anno, mese_num): {dups}")

GROUP BY non trovato — verificare manualmente.
Duplicati su (anno, mese_num): 0



In [ ]:
if "anno" in mart.columns:
    anni_mart = sorted(mart["anno"].unique())
    print(f"Anni nel mart ({len(anni_mart)}): {anni_mart[0]}–{anni_mart[-1]}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")

Anni nel mart (18): 2001–2018

Null per colonna mart:
  nessuno



In [ ]:
# Totali mart vs clean — incidenti, morti, feriti, incidenti_mortali
metrics = ["incidenti", "morti", "feriti", "incidenti_mortali"]
print(f"{'Metrica':<25} {'MART':>12} {'CLEAN':>12} {'Match':>8}")
print("-" * 60)
for m in metrics:
    if m in mart.columns and m in clean.columns:
        tot_m = mart[m].sum()
        tot_c = clean[m].sum()
        match = abs(tot_m - tot_c) < 10
        print(f"{m:<25} {tot_m:>12,.0f} {tot_c:>12,.0f} {'OK' if match else 'DIFF':>8}")

Metrica                           MART        CLEAN    Match
------------------------------------------------------------
incidenti                    3,736,227    3,736,227       OK
morti                           84,263       84,263       OK
feriti                       5,091,330    5,091,330       OK
incidenti_mortali               77,695       77,695       OK



In [ ]:
PERIMETRO = (
    "MIT incidentalità mensile 2001-2018 — incidenti, morti, feriti, incidenti mortali per mese"
)

print(f"Slug         : {SLUG}")
print(f"Anno run     : {ANNO_RUN} (placeholder — dati 2001-2018)")
print(f"Tabella mart : {MART_TABLE}")
print(f"Metrica mart : {METRICA}")
print(f"Metrica clean: {METRICA_CLEAN}")
print(f"Perimetro    : {PERIMETRO}")

Slug         : mit_incidentalita_mensile
Anno run     : 2001 (placeholder — dati 2001-2018)
Tabella mart : mart_mensile
Metrica mart : incidenti
Metrica clean: incidenti
Perimetro    : MIT incidentalità mensile 2001-2018 — incidenti, morti, feriti, incidenti mortali per mese

